# Eksperimen: Prediksi Inflasi Bulanan 5 Kota dengan LSTM

**Afnan Ahsana Syadida** · Skripsi S.Kom., Teknik Informatika, Universitas Muhammadiyah Magelang (2026)

Inflasi bulanan BPS dan UMR untuk Medan, DKI Jakarta, Bandung, Surabaya, dan Makassar (Jan 2000 – Des 2025), diproses sekaligus untuk 5 kota.

| Tahap | Isi |
|---|---|
| **0. Pembanding sederhana** | Bulan lalu, rata-rata 12 bulan, musiman, rata-rata historis |
| **1. Seleksi arsitektur** | 3 konfigurasi LSTM dari penelitian terdahulu, tanpa penalaan |
| **2. Tambah UMR** | Konfigurasi terpilih + UMR (dua versi: level dan pertumbuhan tahunan) |
| **3. Optuna** | Penalaan hiperparameter per kota, untuk kedua versi UMR |

### Cara evaluasi
- Data dibagi **80% latih / 20% uji** secara berurutan (uji: Okt 2020 – Des 2025), sama seperti skripsi.
- Skenario 1 dan 2 memakai jumlah epoch tetap dari artikel acuan, tanpa early stopping.
- Skenario 3: Optuna menilai setiap kombinasi dengan **validasi silang *walk-forward* 3 lipatan yang seluruhnya berada di dalam data latih**. Data uji tidak disentuh sampai evaluasi akhir.

## 0. Setup

In [ ]:
# Jalankan sekali di Google Colab
!pip install -q optuna

In [ ]:
import os, random, warnings, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

tf.get_logger().setLevel("ERROR")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)
set_seed()

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ===================== KONFIGURASI =====================
# Lokasi data: otomatis mencari di repo (../data atau data/), atau file yang di-upload ke Colab
from pathlib import Path
_cands = [Path("../data/inflasi_umr_5kota_2000_2025.csv"), Path("data/inflasi_umr_5kota_2000_2025.csv"), Path("inflasi_umr_5kota_2000_2025.csv")]
DATA_PATH  = next((p for p in _cands if p.exists()), _cands[-1])
OUTPUT_DIR = str(DATA_PATH.parent.parent / "outputs") if DATA_PATH.parent.name == "data" else "outputs"
CITIES = ["KOTA MEDAN", "DKI JAKARTA", "KOTA BANDUNG", "KOTA SURABAYA", "KOTA MAKASSAR"]

TRAIN_RATIO = 0.80                 # 80% latih, 20% uji (sama dengan skripsi)
UMR_MODES   = ["level", "growth"]  # "level" = UMR apa adanya; "growth" = pertumbuhan UMR tahunan (%)

# Optuna (Skenario 3)
N_TRIALS     = 25                  # per kota, per versi UMR
CV_FOLDS     = 3                   # lipatan walk-forward di dalam data latih
CV_FOLD_LEN  = 24                  # panjang tiap lipatan evaluasi (bulan)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# (Opsional, disarankan) Simpan outputs ke Google Drive agar studi Optuna bisa dilanjutkan bila sesi Colab terputus
# from google.colab import drive
# drive.mount("/content/drive")
# OUTPUT_DIR = "/content/drive/MyDrive/skripsi_inflasi/outputs_v2"; os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df["Tanggal"] = pd.to_datetime(df["Tanggal"], format="%m/%d/%Y")
df = df.sort_values(["Kota", "Tanggal"]).reset_index(drop=True)

print(df.groupby("Kota").agg(baris=("Inflasi", "size"), mulai=("Tanggal", "min"), akhir=("Tanggal", "max")))
print("Data kosong:", int(df.isna().sum().sum()))
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
for c in CITIES:
    d = df[df["Kota"] == c]
    ax.plot(d["Tanggal"], d["Inflasi"], linewidth=1, label=c.title())
ax.set_title("Inflasi bulanan (%) — 5 kota, 2000–2025"); ax.set_ylabel("Inflasi (%)")
ax.legend(ncol=5, fontsize=8); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 2. Persiapan data

- `get_city` menyiapkan satu kota beserta fitur UMR sesuai `umr_mode`.
- `prepare` membentuk jendela *sliding window* untuk satu periode latih `[0, train_end)` dan satu periode evaluasi `[train_end, eval_end)`. Scaler **selalu di-fit hanya pada periode latih** tersebut, baik untuk pembagian 80/20 maupun di setiap lipatan validasi silang.

In [ ]:
def get_city(city, umr_mode=None):
    d = df[df["Kota"] == city].sort_values("Tanggal").reset_index(drop=True).copy()
    if umr_mode == "level":
        d["UMR_feat"] = d["UMR"].astype(float)
    elif umr_mode == "growth":
        # pertumbuhan UMR dibanding bulan yang sama tahun lalu (%); tahun 2000 diisi nilai 2001
        d["UMR_feat"] = (d["UMR"] / d["UMR"].shift(12) - 1).bfill() * 100
    return d

def get_scaler(name):
    return {"zscore": StandardScaler(), "minmax": MinMaxScaler(), "robust": RobustScaler()}[name]

def make_windows(values, target, lag, start, end):
    X = np.stack([values[i - lag:i] for i in range(start, end)])
    return X, target[start:end]

def prepare(city, lag, scaler_name, umr_mode, train_end, eval_end):
    d = get_city(city, umr_mode)
    cols = ["Inflasi"] + (["UMR_feat"] if umr_mode else [])
    scalers = {c: get_scaler(scaler_name).fit(d.loc[:train_end - 1, [c]]) for c in cols}
    scaled = np.column_stack([scalers[c].transform(d[[c]]).ravel() for c in cols])
    return {
        "train": make_windows(scaled, scaled[:, 0], lag, lag, train_end),
        "eval":  make_windows(scaled, scaled[:, 0], lag, train_end, eval_end),
        "dates": d["Tanggal"].iloc[train_end:eval_end].values,
        "inv":   lambda z: scalers["Inflasi"].inverse_transform(np.asarray(z).reshape(-1, 1)).ravel(),
    }

N = len(get_city(CITIES[0]))
TRAIN_END = int(N * TRAIN_RATIO)
FOLDS = [(TRAIN_END - (CV_FOLDS - k) * CV_FOLD_LEN, TRAIN_END - (CV_FOLDS - k - 1) * CV_FOLD_LEN) for k in range(CV_FOLDS)]

dates = get_city(CITIES[0])["Tanggal"].dt.strftime("%b %Y").values
print(f"Latih : {dates[0]} – {dates[TRAIN_END-1]} ({TRAIN_END} bulan)")
print(f"Uji   : {dates[TRAIN_END]} – {dates[-1]} ({N-TRAIN_END} bulan)")
for k, (a, b) in enumerate(FOLDS, 1):
    print(f"Lipatan CV {k}: latih s.d. {dates[a-1]}, evaluasi {dates[a]} – {dates[b-1]}")

In [ ]:
# Mengapa dua versi UMR? Nilai UMR level di periode uji berada di luar rentang data latih:
for c in CITIES:
    u = get_city(c, "level")["UMR_feat"].values.reshape(-1, 1)
    z = StandardScaler().fit(u[:TRAIN_END]).transform(u).ravel()
    g = get_city(c, "growth")["UMR_feat"].values
    print(f"{c:14s} UMR level (z-score): latih maks {z[:TRAIN_END].max():.2f}, uji {z[TRAIN_END:].min():.2f}–{z[TRAIN_END:].max():.2f}"
          f" | pertumbuhan: latih {g[:TRAIN_END].min():.1f}–{g[:TRAIN_END].max():.1f}%, uji {g[TRAIN_END:].min():.1f}–{g[TRAIN_END:].max():.1f}%")

## 3. Pembanding sederhana (data uji)

In [ ]:
def metrics(y, p):
    mse = mean_squared_error(y, p)
    return {"MSE": mse, "RMSE": np.sqrt(mse), "MAE": mean_absolute_error(y, p),
            "rasio_variasi": float(np.std(p) / np.std(y))}   # ~0 = prediksi datar, ~1 = variasi setara aktual

results, preds = [], []
def log(scenario, model_name, city, r, params=None, extra=None):
    results.append({"Skenario": scenario, "Model": model_name, "Kota": city, **r["metrics"],
                    **(extra or {}), "Params": json.dumps(params or {})})
    preds.extend({"Skenario": scenario, "Model": model_name, "Kota": city, "Tanggal": t,
                  "Aktual": round(float(a), 4), "Prediksi": round(float(p), 4)}
                 for t, a, p in zip(r["dates"], r["actual"], r["pred"]))

for c in CITIES:
    d = get_city(c); y = d["Inflasi"].values; n = len(y); te = TRAIN_END; yt = y[te:]
    cands = {
        "Naive (bulan lalu)":      y[te - 1:n - 1],
        "Rata-rata historis":      np.full(n - te, y[:te].mean()),
        "Rata-rata 12 bulan":      np.array([y[i - 12:i].mean() for i in range(te, n)]),
        "Musiman (12 bulan lalu)": y[te - 12:n - 12],
    }
    for name, p in cands.items():
        log("S0", name, c, {"metrics": metrics(yt, p), "dates": d["Tanggal"].values[te:], "actual": yt, "pred": p})

pd.DataFrame(results).pivot(index="Kota", columns="Model", values="MSE").round(4)

## 4. Fungsi model

In [ ]:
def build_model(input_shape, cfg):
    layers = [Input(shape=input_shape)]
    units = cfg["units"]
    for i, u in enumerate(units):
        layers.append(LSTM(u, activation=cfg.get("activation", "tanh"), return_sequences=i < len(units) - 1))
        if cfg.get("dropout", 0) > 0:
            layers.append(Dropout(cfg["dropout"]))
    layers.append(Dense(1))
    model = Sequential(layers)
    opt = SGD(learning_rate=cfg["lr"]) if cfg.get("optimizer") == "sgd" else Adam(learning_rate=cfg["lr"])
    model.compile(optimizer=opt, loss="mse")
    return model

def fit_eval(city, cfg, umr_mode, train_end, eval_end):
    # Latih pada [0, train_end) dengan epoch tetap, evaluasi pada [train_end, eval_end)
    set_seed(); tf.keras.backend.clear_session()
    data = prepare(city, cfg["lag"], cfg["scaler"], umr_mode, train_end, eval_end)
    (Xtr, ytr), (Xev, yev) = data["train"], data["eval"]
    model = build_model(Xtr.shape[1:], cfg)
    model.fit(Xtr, ytr, epochs=cfg["epochs"], batch_size=cfg["batch"], shuffle=False, verbose=0)
    p, a = data["inv"](model.predict(Xev, verbose=0)), data["inv"](yev)
    return {"metrics": metrics(a, p), "dates": data["dates"], "actual": a, "pred": p}

def fit_test(city, cfg, umr_mode):
    return fit_eval(city, cfg, umr_mode, TRAIN_END, N)

## 5. Skenario 1 — Seleksi arsitektur

Konfigurasi dari Tabel 3.2 skripsi. Artikel 1 (Muhajir dkk., 2024) tidak mencantumkan jumlah unit dan batch size, jadi dipakai 20 unit dan batch 32.

In [ ]:
CONFIGS_S1 = {
    "Artikel 1 (Muhajir dkk., 2024)": dict(lag=12, units=[20], activation="sigmoid", optimizer="sgd",
                                          lr=0.05, epochs=400, batch=32, scaler="minmax"),  # units & batch tidak disebut di artikel
    "Artikel 2 (Masipupu dkk., 2025)": dict(lag=12, units=[20], activation="tanh", optimizer="adam",
                                           lr=0.001, epochs=200, batch=32, scaler="minmax"),
    "Artikel 3 (Leipary & Setiawan, 2025)": dict(lag=12, units=[256, 128, 64], dropout=0.1, activation="tanh",
                                                optimizer="adam", lr=0.001, epochs=100, batch=16, scaler="zscore"),
}

for name, cfg in CONFIGS_S1.items():
    for c in CITIES:
        r = fit_test(c, cfg, None)
        log("S1", name, c, r, cfg)
        print(f"{name:38s} {c:14s} MSE={r['metrics']['MSE']:.4f}  rasio_variasi={r['metrics']['rasio_variasi']:.2f}")

In [ ]:
s1 = pd.DataFrame([r for r in results if r["Skenario"] == "S1"])
display(s1.pivot(index="Kota", columns="Model", values="MSE").round(4))
display(s1.pivot(index="Kota", columns="Model", values="rasio_variasi").round(2))

In [ ]:
BASELINE_NAME = "Artikel 3 (Leipary & Setiawan, 2025)"   # dipilih karena prediksinya mengikuti pola (rasio variasi), bukan sekadar MSE terkecil
BASE_CFG = CONFIGS_S1[BASELINE_NAME]

p = pd.DataFrame(preds)
fig, axes = plt.subplots(len(CONFIGS_S1), 1, figsize=(12, 9), sharex=True)
for ax, name in zip(axes, CONFIGS_S1):
    d = p[(p.Skenario == "S1") & (p.Model == name) & (p.Kota == "DKI JAKARTA")]
    ax.plot(d.Tanggal, d.Aktual, label="Aktual"); ax.plot(d.Tanggal, d.Prediksi, "--", label="Prediksi")
    ax.set_title(f"{name} — DKI Jakarta"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Skenario 2 — Menambahkan UMR (dua versi)

In [ ]:
for mode in UMR_MODES:
    for c in CITIES:
        r = fit_test(c, BASE_CFG, mode)
        log("S2", f"Baseline + UMR ({mode})", c, r, {**BASE_CFG, "umr_mode": mode})
        print(f"UMR {mode:6s} {c:14s} MSE={r['metrics']['MSE']:.4f}")

## 7. Skenario 3 — Optuna dengan validasi silang di dalam data latih

Setiap trial dilatih dan dinilai pada 3 lipatan *walk-forward*. Nilai objektifnya adalah rata-rata MSE ketiga lipatan. Trial yang jelas buruk dihentikan lebih awal (*pruning*) setelah lipatan pertama atau kedua. Konfigurasi terbaik kemudian dilatih ulang pada seluruh 80% data latih dan diuji **sekali** pada data uji.

Studi disimpan di SQLite, jadi kalau Colab terputus, jalankan ulang sel ini dan Optuna akan melanjutkan.

In [ ]:
def cfg_from(params):
    return dict(scaler=params["scaler"], lag=params["lag"], units=[params["units_l1"], params["units_l2"]],
                lr=params["lr"], batch=params["batch"], epochs=params["epochs"], activation="tanh", optimizer="adam")

def objective_factory(city, umr_mode):
    def objective(trial):
        params = dict(
            scaler   = trial.suggest_categorical("scaler", ["robust", "zscore"]),
            lag      = trial.suggest_int("lag", 3, 24, step=3),
            units_l1 = trial.suggest_categorical("units_l1", [64, 128, 256]),
            units_l2 = trial.suggest_categorical("units_l2", [32, 64, 128]),
            lr       = trial.suggest_float("lr", 1e-4, 1e-2, log=True),
            batch    = trial.suggest_categorical("batch", [16, 32, 64]),
            epochs   = trial.suggest_categorical("epochs", [50, 75, 100, 125, 150]),
        )
        scores = []
        for k, (a, b) in enumerate(FOLDS):
            scores.append(fit_eval(city, cfg_from(params), umr_mode, a, b)["metrics"]["MSE"])
            trial.report(float(np.mean(scores)), k)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores))
    return objective

best_params = {}
for mode in UMR_MODES:
    for c in CITIES:
        study = optuna.create_study(
            study_name=f"v2_{mode}_{c.replace(' ', '_')}", direction="minimize",
            sampler=TPESampler(seed=SEED), pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=0),
            storage=f"sqlite:///{OUTPUT_DIR}/optuna_v2.db", load_if_exists=True)
        remaining = max(0, N_TRIALS - len(study.trials))
        study.optimize(objective_factory(c, mode), n_trials=remaining, show_progress_bar=True)
        best_params[(mode, c)] = {**study.best_params, "best_trial": study.best_trial.number, "cv_MSE": study.best_value}
        print(mode, c, best_params[(mode, c)])

bp_df = pd.DataFrame(best_params).T.rename_axis(["umr_mode", "Kota"])
bp_df.to_csv(f"{OUTPUT_DIR}/optuna_best_params.csv")
bp_df

In [ ]:
for (mode, c), bp in best_params.items():
    cfg = cfg_from(bp)
    r = fit_test(c, cfg, mode)
    log("S3", f"Baseline + UMR ({mode}) + Optuna", c, r, {**cfg, "umr_mode": mode}, {"cv_MSE": bp["cv_MSE"]})
    print(f"UMR {mode:6s} {c:14s} CV MSE={bp['cv_MSE']:.4f}  uji MSE={r['metrics']['MSE']:.4f}")

## 8. Ringkasan hasil (data uji, Okt 2020 – Des 2025)

In [ ]:
res_df = pd.DataFrame(results)
res_df.to_csv(f"{OUTPUT_DIR}/metrics_all.csv", index=False)

pick = {
    "0. Rata-rata 12 bulan":        ("S0", "Rata-rata 12 bulan"),
    "1. Baseline (Leipary)":        ("S1", BASELINE_NAME),
    "2a. + UMR level":              ("S2", "Baseline + UMR (level)"),
    "2b. + UMR growth":             ("S2", "Baseline + UMR (growth)"),
    "3a. + UMR level + Optuna":     ("S3", "Baseline + UMR (level) + Optuna"),
    "3b. + UMR growth + Optuna":    ("S3", "Baseline + UMR (growth) + Optuna"),
}
summary = pd.DataFrame({k: res_df[(res_df.Skenario == s) & (res_df.Model == m)].set_index("Kota")["MSE"]
                        for k, (s, m) in pick.items()}).loc[CITIES].round(4)
for k in ["3a. + UMR level + Optuna", "3b. + UMR growth + Optuna"]:
    summary[f"Δ {k[:2]} vs rata-rata 12 bln (%)"] = ((summary[k] / summary["0. Rata-rata 12 bulan"] - 1) * 100).round(1)
summary.to_csv(f"{OUTPUT_DIR}/summary_test_mse.csv")   # Δ negatif = error lebih kecil dari pembanding
summary

In [ ]:
thesis = pd.DataFrame({
    "Skripsi: Baseline": {"KOTA MEDAN": 0.6047, "DKI JAKARTA": 0.2097, "KOTA BANDUNG": 0.3731, "KOTA SURABAYA": 0.1831, "KOTA MAKASSAR": 0.2701},
    "Skripsi: + UMR":    {"KOTA MEDAN": 0.4677, "DKI JAKARTA": 0.1982, "KOTA BANDUNG": 0.2994, "KOTA SURABAYA": 0.1547, "KOTA MAKASSAR": 0.3351},
    "Skripsi: + Optuna": {"KOTA MEDAN": 0.2712, "DKI JAKARTA": 0.1285, "KOTA BANDUNG": 0.2449, "KOTA SURABAYA": 0.1377, "KOTA MAKASSAR": 0.2613},
})
summary.iloc[:, :6].join(thesis)

In [ ]:
pred_df = pd.DataFrame(preds)
pred_df.to_csv(f"{OUTPUT_DIR}/predictions_test.csv", index=False)

best_mode = summary[["3a. + UMR level + Optuna", "3b. + UMR growth + Optuna"]].mean().idxmin()
best_model = pick[best_mode][1]
fig, axes = plt.subplots(len(CITIES), 1, figsize=(12, 14), sharex=True)
for ax, c in zip(axes, CITIES):
    d = pred_df[(pred_df.Kota == c) & (pred_df.Model == best_model)]
    b = pred_df[(pred_df.Kota == c) & (pred_df.Model == "Rata-rata 12 bulan")]
    ax.plot(pd.to_datetime(d.Tanggal), d.Aktual, label="Aktual", linewidth=1.6)
    ax.plot(pd.to_datetime(d.Tanggal), d.Prediksi, "--", label=best_model)
    ax.plot(pd.to_datetime(b.Tanggal), b.Prediksi, ":", label="Rata-rata 12 bulan")
    ax.set_title(c.title()); ax.grid(alpha=.3)
axes[0].legend(ncol=3, fontsize=8); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/actual_vs_pred_test.png", dpi=150); plt.show()

**File keluaran** di `outputs/`: `metrics_all.csv`, `optuna_best_params.csv`, `summary_test_mse.csv`, `predictions_test.csv`, `actual_vs_pred_test.png`, `optuna_v2.db`.